# 06-01 다음 토큰 예측

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import json
import math
import os
import random
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 말뭉치를 긴 문자열 하나로

[KLUE](https://github.com/KLUE-benchmark/KLUE) NLI 학습 데이터의 문장을 씁니다. (CC BY-SA 4.0)

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")
with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
sentences = sorted({s.strip() for d in data for s in (d["premise"], d["hypothesis"])})
random.seed(0)
random.shuffle(sentences)

text = "\n".join(sentences) + "\n"  # 문장 끝은 줄바꿈 문자로 표시해요
print(len(text))
print(text[:100])

In [ ]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = chars
V = len(chars)
print("어휘집 크기:", V)

data_ids = torch.tensor([stoi[ch] for ch in text])  # (전체 문자 수,)
n = int(len(data_ids) * 0.9)
train_ids, val_ids = data_ids[:n], data_ids[n:]
print(train_ids.shape, val_ids.shape)

## 2. 입력과 정답은 한 칸 차이

In [ ]:
chunk = train_ids[:11]
x, y = chunk[:-1], chunk[1:]  # 입력 10개, 정답 10개
for t in range(5):
    context = "".join(itos[i] for i in x[: t + 1])
    print(f"{context!r:12} -> {itos[y[t]]!r}")

## 3. 가장 단순한 모델: 바로 앞 글자만 보고 세기 (바이그램)

In [ ]:
counts = torch.zeros(V, V)  # (V, V), counts[a, b] = a 다음에 b가 나온 횟수
counts.index_put_((train_ids[:-1], train_ids[1:]), torch.ones(n - 1), accumulate=True)
probs = (counts + 0.01) / (counts + 0.01).sum(dim=1, keepdim=True)  # (V, V), 행마다 합이 1. 확률 0을 피하려고 0.01을 더해요

for ch in ["학", "니", "\n"]:
    top = probs[stoi[ch]].topk(5)
    print(repr(ch), "->", [(itos[i], round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())])

In [ ]:
top_ids = torch.bincount(train_ids, minlength=V).topk(20).indices  # 가장 자주 나온 문자 20개
sub = probs[top_ids][:, top_ids]  # (20, 20)
plt.figure(figsize=(5.5, 4.5))
plt.imshow(sub, cmap="Oranges")
plt.colorbar()
plt.xlabel("next char (index in top-20)")
plt.ylabel("current char (index in top-20)")
plt.title("P(next | current)")
plt.show()
print({i: itos[c] for i, c in enumerate(top_ids.tolist())})

## 4. 손실로 모델 평가하기

In [ ]:
def bigram_loss(ids):
    p = probs[ids[:-1], ids[1:]]  # 정답 문자에 준 확률
    return -torch.log(p).mean().item()

print(f"아무것도 모를 때(균등 분포) 손실: {math.log(V):.3f}")
val_sample = val_ids[:20001]  # 검증 데이터 앞부분 2만 글자로 평가해요
print(f"바이그램 손실 (검증): {bigram_loss(val_sample):.3f}")

## 5. 바이그램으로 글 만들기

In [ ]:
def generate_bigram(start, length=60, seed=0):
    g = torch.Generator().manual_seed(seed)
    out = [stoi[start]]
    for _ in range(length):
        nxt = torch.multinomial(probs[out[-1]], 1, generator=g).item()  # 확률대로 하나 뽑기
        out.append(nxt)
    return "".join(itos[i] for i in out)

print(generate_bigram("오"))

## 6. 같은 모델을 신경망으로: 임베딩 하나

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.table = nn.Embedding(V, V)  # 행 = 현재 문자, 열 = 다음 문자 점수

    def forward(self, ids):
        return self.table(ids)  # (B, T) -> (B, T, V)

def get_batch(ids, batch_size, block_size):
    starts = torch.randint(0, len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([ids[s : s + block_size] for s in starts])  # (B, block_size)
    y = torch.stack([ids[s + 1 : s + block_size + 1] for s in starts])  # (B, block_size), 한 칸 뒤
    return x.to(device), y.to(device)

torch.manual_seed(0)
model = BigramLM(V).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.1)
losses = []
for step in range(1500):
    x, y = get_batch(train_ids, 64, 32)
    logits = model(x)  # (64, 32, V)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

with torch.no_grad():
    x, y = val_sample[:-1].to(device), val_sample[1:].to(device)
    val_loss = F.cross_entropy(model(x), y).item()
print(f"신경망 바이그램 손실 (검증): {val_loss:.3f}")

In [ ]:
plt.figure(figsize=(6, 3.3))
plt.plot(losses, color="tab:red", alpha=0.6, label="neural bigram (train)")
plt.axhline(bigram_loss(val_sample), color="k", ls="--", label="count bigram (val)")
plt.axhline(math.log(V), color="gray", ls=":", label="uniform")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()